<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Fraud-Detection-with-Federated-Learning/blob/main/PaySIm_Fraud_Detection_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os, random, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import kagglehub

# ========================= SEED =========================
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ========================= GPU (Colab Pro T4) =========================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU Detected - Using T4 High-RAM")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("⚠️ No GPU - Running on CPU")

# ========================= LOAD & REDUCE PaySim =========================
print("\n=== LOADING PaySim ===")
path = kagglehub.dataset_download("ealaxi/paysim1")
df = pd.read_csv(path + "/PS_20174392719_1491204439457_log.csv")

print("ORIGINAL DATASET")
print(f"Total samples: {len(df):,}")
print(f"Fraud: {df['isFraud'].sum():,}")
print(f"Non-Fraud: {len(df) - df['isFraud'].sum():,}")

# KEEP ALL FRAUD + REDUCE NON-FRAUD TO 1 MILLION
fraud_df = df[df["isFraud"] == 1]
nonfraud_df = df[df["isFraud"] == 0].sample(n=1_000_000, random_state=SEED)
df = pd.concat([fraud_df, nonfraud_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nAFTER REDUCTION (keep all fraud)")
print(f"Total samples: {len(df):,}")
print(f"Fraud: {df['isFraud'].sum():,}")
print(f"Non-Fraud: {len(df) - df['isFraud'].sum():,}")

# ========================= PREPROCESS =========================
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["type"] = le.fit_transform(df["type"])

features = ["step", "type", "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
X = df[features].values.astype("float32")
y = df["isFraud"].values.astype("float32")

# ========================= IID TRAIN-TEST SPLIT =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("\nIID GLOBAL TEST SET")
print(f"Test samples: {len(X_test):,} | Fraud: {int(y_test.sum()):,} | Non-Fraud: {len(y_test)-int(y_test.sum()):,}")

# ========================= NON-IID DIRICHLET SPLIT (12 clients) =========================
NUM_CLIENTS = 12
alpha = 0.5

def dirichlet_split(X, y, num_clients, alpha):
    clients = [[] for _ in range(num_clients)]
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (proportions / proportions.sum() * len(idx)).astype(int)
        proportions[-1] = len(idx) - proportions[:-1].sum()
        start = 0
        for i in range(num_clients):
            end = start + proportions[i]
            clients[i].extend(idx[start:end])
            start = end
    return {i: {"X": X[np.array(clients[i])], "y": y[np.array(clients[i])]} for i in range(num_clients)}

clients_train = dirichlet_split(X_train, y_train, NUM_CLIENTS, alpha)
clients_test = {i: {"X": X_test, "y": y_test} for i in range(NUM_CLIENTS)}

print("\nNON-IID CLIENT DISTRIBUTION (Training)")
for i in range(NUM_CLIENTS):
    yc = clients_train[i]["y"]
    print(f"Client {i+1:2d}: Total={len(yc):,} | Fraud={int(yc.sum()):,} | Non-Fraud={len(yc)-int(yc.sum()):,}")

# ========================= MODEL =========================
def create_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(X.shape[1],)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

LOCAL_EPOCHS = 5
BATCH_SIZE = 1024
NUM_ROUNDS = 20

# ========================= NO FEDERATED LEARNING =========================
print("\n=== NO FEDERATED LEARNING ===")
client_models_noFL = []
noFL_metrics = []

for i in range(NUM_CLIENTS):
    print(f"Training Client {i+1}")
    model = create_model()
    model.fit(clients_train[i]["X"], clients_train[i]["y"],
              epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    client_models_noFL.append(model)

    # Evaluate on global test
    pred = (model.predict(clients_test[i]["X"], verbose=0) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    noFL_metrics.append([acc, prec, rec, f1])
    print(f"  Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

# Save per-client CSV
df_noFL = pd.DataFrame(noFL_metrics, columns=["Accuracy", "Precision", "Recall", "F1"])
df_noFL.to_csv("paysim_noFL_metrics.csv", index=False)
print("NoFL Avg ->", df_noFL.mean().round(4))

# ========================= FEDAVG (Improved with final average) =========================
print("\n=== FEDAVG ===")
global_model_fedavg = create_model()
round_metrics_fedavg = []   # list to store all 20 rounds

def fedavg_aggregate(weights_list):
    return [np.mean(w, axis=0) for w in zip(*weights_list)]

for r in range(NUM_ROUNDS):
    client_weights = []
    for i in range(NUM_CLIENTS):
        local_model = create_model()
        local_model.set_weights(global_model_fedavg.get_weights())
        local_model.fit(clients_train[i]["X"], clients_train[i]["y"],
                        epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
        client_weights.append(local_model.get_weights())

    global_model_fedavg.set_weights(fedavg_aggregate(client_weights))

    # Evaluate on global IID test set
    pred = (global_model_fedavg.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    loss = global_model_fedavg.evaluate(X_test, y_test, verbose=0)[0]

    round_metrics_fedavg.append([acc, prec, rec, f1])
    print(f"Round {r+1:2d} → Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Loss: {loss:.4f}")

# === FINAL AVERAGE FOR FEDAVG ===
avg_fedavg = np.mean(round_metrics_fedavg, axis=0)
print("\n=== FEDAVG FINAL AVERAGE (across 20 rounds) ===")
print(f"Avg Accuracy : {avg_fedavg[0]:.4f}")
print(f"Avg Precision: {avg_fedavg[1]:.4f}")
print(f"Avg Recall   : {avg_fedavg[2]:.4f}")
print(f"Avg F1-score : {avg_fedavg[3]:.4f}")

pd.DataFrame(round_metrics_fedavg, columns=["Accuracy","Precision","Recall","F1"]).to_csv("paysim_fedavg_metrics.csv", index=False)

# ========================= FEDBN (Improved with final average) =========================
print("\n=== FEDBN ===")
global_model_fedbn = create_model()
client_models_fedbn = [create_model() for _ in range(NUM_CLIENTS)]
for m in client_models_fedbn:
    m.set_weights(global_model_fedbn.get_weights())

def sync_non_bn(global_model, local_model):
    for g_layer, l_layer in zip(global_model.layers, local_model.layers):
        if "batch_normalization" not in g_layer.name.lower():
            l_layer.set_weights(g_layer.get_weights())

def fedbn_aggregate(client_models, global_model):
    new_weights = []
    for i, layer in enumerate(global_model.layers):
        if "batch_normalization" not in layer.name.lower():
            weights = [m.layers[i].get_weights() for m in client_models]
            avg = [np.mean(w, axis=0) for w in zip(*weights)]
            new_weights.extend(avg)
        else:
            new_weights.extend(layer.get_weights())
    return new_weights

round_metrics_fedbn = []

for r in range(NUM_ROUNDS):
    for i in range(NUM_CLIENTS):
        sync_non_bn(global_model_fedbn, client_models_fedbn[i])
        client_models_fedbn[i].fit(clients_train[i]["X"], clients_train[i]["y"],
                                   epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    global_model_fedbn.set_weights(fedbn_aggregate(client_models_fedbn, global_model_fedbn))

    # Evaluate average across 12 client models (FedBN keeps local BN)
    accs, precs, recs, f1s = [], [], [], []
    for m in client_models_fedbn:
        pred = (m.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
        accs.append(accuracy_score(y_test, pred))
        precs.append(precision_score(y_test, pred, zero_division=0))
        recs.append(recall_score(y_test, pred, zero_division=0))
        f1s.append(f1_score(y_test, pred, zero_division=0))

    avg_acc = np.mean(accs)
    avg_prec = np.mean(precs)
    avg_rec = np.mean(recs)
    avg_f1 = np.mean(f1s)

    round_metrics_fedbn.append([avg_acc, avg_prec, avg_rec, avg_f1])
    print(f"Round {r+1:2d} → Avg Acc: {avg_acc:.4f} | Prec: {avg_prec:.4f} | Rec: {avg_rec:.4f} | F1: {avg_f1:.4f}")

# === FINAL AVERAGE FOR FEDBN ===
avg_fedbn = np.mean(round_metrics_fedbn, axis=0)
print("\n=== FEDBN FINAL AVERAGE (across 20 rounds) ===")
print(f"Avg Accuracy : {avg_fedbn[0]:.4f}")
print(f"Avg Precision: {avg_fedbn[1]:.4f}")
print(f"Avg Recall   : {avg_fedbn[2]:.4f}")
print(f"Avg F1-score : {avg_fedbn[3]:.4f}")

pd.DataFrame(round_metrics_fedbn, columns=["Accuracy","Precision","Recall","F1"]).to_csv("paysim_fedbn_metrics.csv", index=False)
print("\n PAYSim DONE! All CSVs saved.")

✅ GPU Detected - Using T4 High-RAM

=== LOADING PaySim ===
Using Colab cache for faster access to the 'paysim1' dataset.
ORIGINAL DATASET
Total samples: 6,362,620
Fraud: 8,213
Non-Fraud: 6,354,407

AFTER REDUCTION (keep all fraud)
Total samples: 1,008,213
Fraud: 8,213
Non-Fraud: 1,000,000

IID GLOBAL TEST SET
Test samples: 201,643 | Fraud: 1,643 | Non-Fraud: 200,000

NON-IID CLIENT DISTRIBUTION (Training)
Client  1: Total=12,997 | Fraud=80 | Non-Fraud=12,917
Client  2: Total=108,847 | Fraud=299 | Non-Fraud=108,548
Client  3: Total=605 | Fraud=221 | Non-Fraud=384
Client  4: Total=35,902 | Fraud=142 | Non-Fraud=35,760
Client  5: Total=2,404 | Fraud=2,370 | Non-Fraud=34
Client  6: Total=53,665 | Fraud=496 | Non-Fraud=53,169
Client  7: Total=15,678 | Fraud=747 | Non-Fraud=14,931
Client  8: Total=173,778 | Fraud=411 | Non-Fraud=173,367
Client  9: Total=14,250 | Fraud=1,289 | Non-Fraud=12,961
Client 10: Total=24,708 | Fraud=351 | Non-Fraud=24,357
Client 11: Total=55,951 | Fraud=126 | Non-Fra